In [ ]:
from dataloader import build_loaders
from model import build_model

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)

# IMPORTANT: Samples=5 (the 5 frequency bands), not 200
model = build_model(nb_classes=4, Chans=62, Samples=5)

run_name = "TestRun"
leave_one_out = False

In [ ]:
train_loader, val_loader = build_loaders(
    data_root   = '/fs/vulcan-projects/fsh_track/jason-bhargav-temp/CMSC472-Final/data',
    dataset     = 'SEED-IV',
    window_sec  = 1.0,
    sfreq       = 200,
    n_per_class = 8,

    leave_one_out = leave_one_out,
    # val_subject = 1,
    val_fraction = 0.2,
)

In [ ]:
import torch
from model import build_model, EEGDataset, BalancedBatchSampler, Trainer
from losses import ClassificationLoss, ContrastiveLoss, ContrastivePrototype, LeaveOneOutContrastiveLearning

model     = build_model(nb_classes=4, Chans=62, Samples=5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
warmup_epochs=5

trainer = Trainer(
    
    # '''None'''
    model, ClassificationLoss(), None,
    optimizer, lambda_con=0.5, warmup_epochs=warmup_epochs, device='cuda'

    # '''Contrastive Loss'''
    # model, ClassificationLoss(), ContrastiveLoss(temperature=0.1),
    # optimizer, lambda_con=0.5, warmup_epochs=5, device='cuda'

    # '''Contrastive Prototype'''
    # model, ClassificationLoss(), ContrastivePrototype(num_classes=4),
    # optimizer, lambda_con=0.5, warmup_epochs=warmup_epochs, device='cuda'

    # '''Contrastive Prototype'''
    # model, ClassificationLoss(), LeaveOneOutContrastiveLearning(temperature=0.1),
    # optimizer, lambda_con=0.5, warmup_epochs=warmup_epochs, device='cuda'
)
history = trainer.fit(train_loader, val_loader, epochs=50)

In [ ]:
import matplotlib.pyplot as plt
import os

def plot_history(history):
    epochs = range(warmup_epochs + 1, len(history['train']) + 1)

    train_loss     = [e['loss']     for e in history['train']][warmup_epochs:]
    val_loss       = [e['loss']     for e in history['val']][warmup_epochs:]
    train_cls_loss = [e['cls_loss'] for e in history['train']][warmup_epochs:]
    val_cls_loss   = [e['cls_loss'] for e in history['val']][warmup_epochs:]
    train_con_loss = [e['con_loss'] for e in history['train']][warmup_epochs:]
    val_con_loss   = [e['con_loss'] for e in history['val']][warmup_epochs:]
    train_acc      = [e['acc']      for e in history['train']][warmup_epochs:]
    val_acc        = [e['acc']      for e in history['val']][warmup_epochs:]

    fig, axes = plt.subplots(1, 4, figsize=(22, 4))
    ax1, ax2, ax3, ax4 = axes

    # Total Loss
    ax1.plot(epochs, train_loss, label='Train Loss')
    ax1.plot(epochs, val_loss,   label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Total Loss')
    ax1.legend()
    ax1.grid(True)

    # cls_loss
    ax2.plot(epochs, train_cls_loss, label='Train cls_loss')
    ax2.plot(epochs, val_cls_loss,   label='Val cls_loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Classification Loss')
    ax2.legend()
    ax2.grid(True)

    # con_loss
    ax3.plot(epochs, train_con_loss, label='Train con_loss')
    ax3.plot(epochs, val_con_loss,   label='Val con_loss')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Loss')
    ax3.set_title('Contrastive Loss')
    ax3.legend()
    ax3.grid(True)

    # Accuracy
    ax4.plot(epochs, train_acc, label='Train Acc')
    ax4.plot(epochs, val_acc,   label='Val Acc')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Accuracy')
    ax4.set_title('Accuracy')
    ax4.legend()
    ax4.grid(True)

    plt.tight_layout()

    save_path = f'checkpoints/{run_name}/plots.png'
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150)
    plt.show()

plot_history(history)

In [ ]:
import os

# Save
save_path = f'checkpoints/{run_name}/model.pt'
os.makedirs('checkpoints', exist_ok=True)

torch.save({
    'model_state_dict'    : model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history'             : history,
}, save_path)

print(f"Model saved to {save_path}")


In [ ]:
checkpoint = torch.load(f'checkpoints/{run_name}/model.pt', map_location='cuda')

model = build_model(nb_classes=4, Chans=62, Samples=5)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_normalized_confusion_matrix(model, loader, device='cuda', class_names=None):
    if class_names is None:
        class_names = ['Neutral', 'Sad', 'Fear', 'Happy']

    # FIX: Ensure model is on the same device as the data
    model.to(device)
    model.eval()
    
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            # Handle cases where batch might have 2 or 3 items
            x, labels = batch[0], batch[1]
            
            # Move data to GPU
            x, labels = x.to(device), labels.to(device)
            
            logits, _ = model(x)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ... rest of the plotting code remains the same ...
    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', values_format='.2f', colorbar=True)
    ax.set_title('Normalized Confusion Matrix')

    os.makedirs(os.path.dirname(f'checkpoints/{run_name}/confusion_matrix.png'), exist_ok=True)
    plt.savefig(f'checkpoints/{run_name}/confusion_matrix.png', dpi=150, bbox_inches='tight')

    plt.show()

# Run it
plot_normalized_confusion_matrix(model, val_loader, device='cuda')
